In [ ]:
import pandas as pd

# Read the parquet file
df_parquet = pd.read_parquet("/users/david/building-data-pipelines/data/yellow_tripdata_2025-01.parquet")

# Display the first five rows
df_parquet.head()

df_parquet.sample(5000).to_csv("yellow_tripdata_sample.csv", index=False)

In [ ]:
# Import API-related Python modules
import json
import certifi
import urllib3
from urllib3 import request

url = "https://data.cityofnewyork.us/resource/gkne-dk5s.json?$limit=5"

# Initialize HTTPS connection
http = urllib3.PoolManager(cert_reqs='CERT_REQUIRED', ca_certs=certifi.where())

# Make the reuqest
response = http.request('GET', url)

# Check status
print(response.status)

if response.status == 200:
    print("Successful connection")
else:
    print(f"Connection failed. Status code: {response.status}")

data = json.loads(response.data.decode('utf-8'))

print(data[0])




200
Successful connection
{'vendor_id': 'CMT', 'pickup_datetime': '2014-11-23T20:31:29.000', 'dropoff_datetime': '2014-11-23T20:31:29.000', 'passenger_count': '3', 'trip_distance': '0', 'pickup_longitude': '0', 'pickup_latitude': '0', 'store_and_fwd_flag': 'N', 'dropoff_longitude': '0', 'dropoff_latitude': '0', 'payment_type': 'CSH', 'fare_amount': '3', 'mta_tax': '0.5', 'tip_amount': '0', 'tolls_amount': '0', 'total_amount': '4', 'imp_surcharge': '0.5', 'rate_code': '1'}


In [ ]:
import sqlite3
import os

print(os.path.exists("/users/david/building-data-pipelines/data/movies.sqlite3"))

# Connect to your SQLite database
conn = sqlite3.connect("movies.sqlite3")

# Create a cursor
cursor = conn.cursor()

# Query all table names
cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")

# Fetch and print
tables = cursor.fetchall()
print("Tables in movies.sqlite3:")
for t in tables:
    print("-", t[0])

conn.close()

True
Tables in movies.sqlite3:


In [ ]:
url = "https://www.fdic.gov/bank-failures/failed-bank-list/"
tables = pd.read_html(url)
print(len(tables))
df = tables[0]
df.head()

1


,Bank Name,City,State,Cert,Acquiring Institution,Closing Date,Fund Sort ascending
0,The Santa Anna National Bank,Santa Anna,Texas,5520,Coleman County State Bank,"June 27, 2025",10549
1,Pulaski Savings Bank,Chicago,Illinois,28611,Millennium Bank,"January 17, 2025",10548
2,The First National Bank of Lindsay,Lindsay,Oklahoma,4134,First Bank & Trust Co.,"October 18, 2024",10547
3,Republic First Bank dba Republic Bank,Philadelphia,Pennsylvania,27332,"Fulton Bank, National Association","April 26, 2024",10546
4,Citizens Bank,Sac City,Iowa,8758,Iowa Trust & Savings Bank,"November 3, 2023",10545


In [ ]:
# Import modules
import json
import sqlite3
import certifi
import pandas as pd
import urllib3
from urllib3 import PoolManager
from bs4 import BeautifulSoup 

# --- CSV ---
def import_csv(filepath:str) -> pd.DataFrame:
    try:
        df = pd.read_csv(filepath)
        print(f"Loaded CSV -> {filepath} ({df.shape[0]} rows)")
        return df
    except Exception as e:
        print(f"CSV import failed: {e}")
        return pd.DataFrame()
    
# --- Parquet ---
def import_parquet(filepath:str) -> pd.DataFrame:
    try:
        df = pd.read_parquet(filepath)
        print(f"Loaded Parquet -> {filepath} {df.shape[0]} rows)")
        return df
    except Exception as e:
        print(f"Parquet import failed: {e}")
        return pd.DataFrame()
    
# --- API (JSON) ---
def import_api_json(url:str) -> pd.DataFrame:
    try:
        http = PoolManager(cert_reqs='CERT_REQUIRED', ca_certs=certifi.where())
        response = http.request('GET', url)
        if response.status != 200:
            print(f"API call failed with HTTP status: {response.status}")
            return pd.DataFrame()
        data = json.loads(response.data.decode("utf-8"))
        df = pd.DataFrame(data)
        print(f"Loaded API JSON -> {url} ({df.shape[0]} rows)")
        return df
    except Exception as e:
        print(f"API JSON import failed: {e}")
        return pd.DataFrame
        
def import_sqlite(db_path: str, table_or_query: str) -> pd.DataFrame:
    try:
        conn = sqlite3.connect(db_path)
        cursor = conn.cursor()

        # Check if table_or_query is actually a table name
        cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
        tables = [t[0] for t in cursor.fetchall()]

        if table_or_query in tables:
            sql = f"SELECT * FROM {table_or_query}"
        else:
            sql = table_or_query  # assume it's a full SQL query

        df = pd.read_sql_query(sql, conn)
        conn.close()

        print(f"Loaded SQLite data ({df.shape[0]} rows, {df.shape[1]} columns)")
        return df

    except Exception as e:
        print(f"SQLite import failed: {e}")
        return pd.DataFrame()

# --- Web Page (HTML Table) ---
def import_web_table(url: str) -> pd.DataFrame:
    import pandas as pd
    import requests
    from bs4 import BeautifulSoup

    # First try pandas.read_html (fast path)
    try:
        tables = pd.read_html(url)
        if tables:
            df = tables[0]
            print(f"Loaded HTML table from {url} ({df.shape[0]} rows)")
            return df
    except Exception:
        pass  # fall through to manual fetch

    # Fallback: requests + BeautifulSoup + read_html on response
    try:
        headers = {"User-Agent": "Mozilla/5.0"}
        resp = requests.get(url, headers=headers, timeout=20)
        resp.raise_for_status()
        tables = pd.read_html(resp.text)
        if tables:
            df = tables[0]
            print(f"Loaded HTML table from {url} ({df.shape[0]} rows)")
            return df
    except Exception:
        pass

    # Deep fallback: manual parse with BeautifulSoup
    try:
        soup = BeautifulSoup(resp.text, "lxml")
        tbls = soup.find_all("table")
        for tbl in tbls:
            ths = [th.get_text(strip=True) for th in tbl.find_all("th")]
            if ths and any("Bank" in h for h in ths):
                rows = []
                for tr in tbl.find_all("tr"):
                    tds = [td.get_text(strip=True) for td in tr.find_all("td")]
                    if tds:
                        rows.append(tds)
                if rows:
                    df = pd.DataFrame(rows, columns=ths[:len(rows[0])])
                    print(f"Loaded HTML table from {url} ({df.shape[0]} rows) [manual parse]")
                    return df
    except Exception as e:
        print(f"Web import failed: {e}")

    print("No HTML tables found")
    return pd.DataFrame()
                
# Universal import wrapper
def import_all_data(
    csv_path: str,
    parquet_path: str,
    api_url: str, 
    db_path: str,
    table_name: str,
    webpage_url: str,
    ) -> dict:
        """
        Returns a dictionary of DataFrames for all data sources.
        """
        data_sources = {
            "csv": import_csv(csv_path),
            "parquet": import_parquet(parquet_path),
            "api": import_api_json(api_url),
            "sqlite": import_sqlite(db_path, table_name),
            "web": import_web_table(webpage_url),
            }
        return data_sources

In [ ]:
# Test functions
csv_file = "/users/david/building-data-pipelines/chapter_4/data/yellow_tripdata_sample.csv"
parquet_file = "/users/david/building-data-pipelines/chapter_4/yellow_tripdata_2025-01.parquet"
api_endpoint = "https://data.cityofnewyork.us/resource/gkne-dk5s.json"
sqlite_file = "/users/david/building-data-pipelines/chapter_4/data/movies.sqlite"
sqlite_table = "movies"
webpage = "https://www.fdic.gov/resources/resolutions/bank-failures/failed-bank-list/"


data_dict = import_all_data(
    csv_file,
    parquet_file,
    api_endpoint,
    sqlite_file,
    sqlite_table,
    webpage,
)

# Display all shapes
for name, df in data_dict.items():
    print(f"{name.upper()} → {df.shape}")

Loaded CSV -> yellow_tripdata_sample.csv (5000 rows)
Loaded Parquet -> yellow_tripdata_2025-01.parquet 3475226 rows)
Loaded API JSON -> https://data.cityofnewyork.us/resource/gkne-dk5s.json (1000 rows)
Loaded SQLite data (4773 rows, 13 columns)
Loaded HTML table from https://www.fdic.gov/resources/resolutions/bank-failures/failed-bank-list/ (25 rows)
CSV → (5000, 20)
PARQUET → (3475226, 20)
API → (1000, 18)
SQLITE → (4773, 13)
WEB → (25, 7)
